# Velocity Model Building Demo
Builds a time velocity model from `velan` picks with the Seismic Unix strategy (`unisam` along time, then `unisam2` across CDPs), smooths it (`smooth2` damped least squares or a 2D moving average) and converts it to interval velocity in depth (`velconv`).

The SU comparison of every step lives in [velocityModelBenchmark.ipynb](velocityModelBenchmark.ipynb).

In [ ]:
%pip install bokeh segyio

# *** Install pyseiskit from PyPI
# %pip install pyseiskit
# *** Install pyseiskit for local development
# %pip install -e ..
!uv pip install -e .. --reinstall

## 1. Data Setup
Needs a `.su` dataset and its `velan` picks file (`cdp=`, then `tnmo=`/`vnmo=` per CDP) in this folder. The dataset is only read for its geometry (CDP range and time sampling).

In [ ]:
import segyio
import numpy as np
from velanPicksReader import readVelanPicks

# Local files only: keep them in the 'demos' folder, next to this notebook
datasetFilename = 'line-1-line-2-data001_crsRegGathers_cdps.su'
picksFilename = 'line-1-line-2-data001_crsRegGathers_cdps.picksvelan.dat'

velocityPicks = readVelanPicks(picksFilename)
pickedCdps = np.array(sorted(velocityPicks))
pickingInterval = int(pickedCdps[1] - pickedCdps[0])

# Model geometry from the dataset headers: one column per CDP, same time sampling as the data
with segyio.su.open(datasetFilename, "r", ignore_geometry=True, endian='little') as seismicDataset:
    datasetCdps = seismicDataset.attributes(segyio.TraceField.CDP)[:]
    numberSamples = len(seismicDataset.samples)
    intervalTimeSamples = seismicDataset.header[0][segyio.TraceField.TRACE_SAMPLE_INTERVAL] / 1e6

cdps = np.arange(datasetCdps.min(), datasetCdps.max() + 1)
timeSamples = np.arange(numberSamples) * intervalTimeSamples

print(f"{len(pickedCdps)} picked CDPs from {pickedCdps[0]} to {pickedCdps[-1]}, every {pickingInterval} CDPs.")
print(f"Model: {numberSamples} samples every {intervalTimeSamples} s, CDPs {cdps[0]} to {cdps[-1]}.")

from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import row
from bokeh.models import LinearColorMapper, ColorBar
from pyseiskit import velocityModel
from pyseiskit import PALETTES
output_notebook()

def plotVelocityModel(model, title, cdps, verticalSamples, velocityRange, verticalLabel="Time (s)", width=420):
    plotFigure = figure(
        width=width,
        height=500,
        title=title,
        x_axis_label="CDP",
        y_axis_label=verticalLabel,
        x_range=(cdps[0], cdps[-1]),
        y_range=(verticalSamples[-1], verticalSamples[0])
    )

    colorMapper = LinearColorMapper(palette=PALETTES['jet'], low=velocityRange[0], high=velocityRange[1])
    plotFigure.image(
        image=[model],
        x=cdps[0],
        y=verticalSamples[0],
        dw=cdps[-1] - cdps[0],
        dh=verticalSamples[-1] - verticalSamples[0],
        color_mapper=colorMapper
    )

    plotFigure.add_layout(ColorBar(color_mapper=colorMapper, title="m/s"), 'right')
    return plotFigure

def plotVelocityFunctions(velocityFunctions, title, timeSamples, picks=None, width=420):
    plotFigure = figure(
        width=width,
        height=500,
        title=title,
        x_axis_label="Velocity (m/s)",
        y_axis_label="Time (s)",
        y_range=(timeSamples[-1], timeSamples[0])
    )
    colors = ['black', 'red', 'blue', 'green', 'orange']
    for (label, velocityFunction), color in zip(velocityFunctions.items(), colors):
        plotFigure.line(velocityFunction, timeSamples, legend_label=label, color=color, line_width=1.5)
    if picks is not None:
        plotFigure.scatter(picks['velocity'], picks['times'], legend_label='picks', color='black', size=7)
    plotFigure.legend.location = 'bottom_left'
    return plotFigure

## 2. Build the Velocity Model
Same as `unisam method=linear` for each picked CDP, then `unisam2 method1=linear` across CDPs: picks are linearly interpolated, and CDPs outside the picked range repeat the nearest picked CDP.

In [ ]:
rmsVelocityModel = velocityModel.buildVelocityModel(
    velocityPicks,
    cdps,
    numberSamples,
    intervalTimeSamples,
    timeMethod='linear',
    cdpMethod='linear'
)
velocityRange = (rmsVelocityModel.min(), rmsVelocityModel.max())

modelFigure = plotVelocityModel(rmsVelocityModel, "RMS Velocity (linear)", cdps, timeSamples, velocityRange, width=800)
pickedTimes = [time for cdp in pickedCdps for time in velocityPicks[cdp]['times']]
pickedTimeCdps = [cdp for cdp in pickedCdps for _ in velocityPicks[cdp]['times']]
modelFigure.scatter(pickedTimeCdps, pickedTimes, color='white', size=3)
show(modelFigure)

## 3. Interpolation Methods
`unisam` methods along time for a single CDP. The cubic methods pass through every pick and, outside the picked times, extrapolate their end polynomials (as SU does), while `linear` repeats the end picks.

The same methods are available across CDPs (`cdpMethod`). Outside the picked CDP range, prefer `linear`: extrapolated cubic polynomials can reach unphysical (even negative) velocities.

In [ ]:
selectedCdp = int(pickedCdps[len(pickedCdps) // 2])
selectedPicks = velocityPicks[selectedCdp]

velocityFunctions = {
    method: velocityModel.interpolateVelocityFunction(
        selectedPicks['times'],
        selectedPicks['velocity'],
        numberSamples,
        intervalTimeSamples,
        method
    )
    for method in ['linear', 'mono', 'akima', 'spline']
}

show(plotVelocityFunctions(velocityFunctions, f"Interpolation methods at CDP {selectedCdp}", timeSamples, selectedPicks, width=600))

## 4. Smoothing
### 4A. Damped Least Squares (SU `smooth2`)
The usual SU route: `smooth2 r1= r2=` on the whole model. Larger values smooth more; SU recommends 1 to 20.

### 4B. Moving Average on the Whole Model
A box of `timeWindow` seconds x `cdpWindow` CDPs, centred on each sample and truncated at the model edges.

### 4C. Moving Average on the Picked CDPs, then Across CDPs
Smooths only the picked velocity functions (one column every `pickingInterval` CDPs) and interpolates across CDPs afterwards, so `cdpWindow` counts picked CDPs.

In [ ]:
# 4A. SU smooth2 r1=10 r2=20
dampedLeastSquaresModel = velocityModel.applyDampedLeastSquaresSmoothing(
    rmsVelocityModel,
    timeSmoothing=10,
    cdpSmoothing=20
)

# 4B. 0.2 s x 201 CDPs box on the whole model
movingAverageModel = velocityModel.applyMovingAverageSmoothing(
    rmsVelocityModel,
    timeWindow=0.2,
    cdpWindow=201,
    intervalTimeSamples=intervalTimeSamples
)

# 4C. 0.2 s x 3 picked CDPs box, then linear interpolation across CDPs
pickedVelocityFunctions = np.stack(
    [
        velocityModel.interpolateVelocityFunction(
            velocityPicks[cdp]['times'],
            velocityPicks[cdp]['velocity'],
            numberSamples,
            intervalTimeSamples
        )
        for cdp in pickedCdps
    ],
    axis=1
)
smoothedPickedFunctions = velocityModel.applyMovingAverageSmoothing(
    pickedVelocityFunctions,
    timeWindow=0.2,
    cdpWindow=3 * pickingInterval,
    intervalTimeSamples=intervalTimeSamples,
    intervalCdps=pickingInterval
)
pickedMovingAverageModel = velocityModel.interpolateAcrossCdps(smoothedPickedFunctions, pickedCdps, cdps)

show(
    row(
        plotVelocityModel(dampedLeastSquaresModel, "Damped Least Squares (smooth2)", cdps, timeSamples, velocityRange),
        plotVelocityModel(movingAverageModel, "Moving Average (whole model)", cdps, timeSamples, velocityRange),
        plotVelocityModel(pickedMovingAverageModel, "Moving Average (picked CDPs)", cdps, timeSamples, velocityRange)
    )
)

In [ ]:
selectedColumn = int(np.searchsorted(cdps, selectedCdp))

show(plotVelocityFunctions(
    {
        'linear': rmsVelocityModel[:, selectedColumn],
        'smooth2': dampedLeastSquaresModel[:, selectedColumn],
        'moving average (whole model)': movingAverageModel[:, selectedColumn],
        'moving average (picked CDPs)': pickedMovingAverageModel[:, selectedColumn],
    },
    f"Smoothing at CDP {selectedCdp}",
    timeSamples,
    selectedPicks,
    width=600
))

## 5. Interval Velocity in Depth (SU `velconv`)
Dix conversion of the smoothed RMS model (`velconv intype=vrmst outtype=vintt`), then time to depth (`velconv intype=vintt outtype=vintz`). Dix amplifies RMS velocity kinks, so it should run on a smoothed model.

In [ ]:
intervalVelocityModel = velocityModel.convertRmsToIntervalVelocity(dampedLeastSquaresModel, intervalTimeSamples)

intervalDepthSamples = 10.0
numberDepthSamples = 800
depthSamples = np.arange(numberDepthSamples) * intervalDepthSamples
depthVelocityModel = velocityModel.convertTimeToDepthVelocity(
    intervalVelocityModel,
    intervalTimeSamples,
    numberDepthSamples,
    intervalDepthSamples
)

intervalVelocityRange = (intervalVelocityModel.min(), intervalVelocityModel.max())
show(
    row(
        plotVelocityModel(dampedLeastSquaresModel, "RMS Velocity (smooth2)", cdps, timeSamples, velocityRange),
        plotVelocityModel(intervalVelocityModel, "Interval Velocity in Time", cdps, timeSamples, intervalVelocityRange),
        plotVelocityModel(depthVelocityModel, "Interval Velocity in Depth", cdps, depthSamples, intervalVelocityRange, verticalLabel="Depth (m)")
    )
)